# Avance 3 — Equipo #48

![Tecnológico de Monterrey](https://i.imgur.com/tHk1B4P.png)

**Febrero 01, 2026**

| Nombre | Matrícula |
|--------|-----------|
| Jorge Daniel Amezola González | A01793759 |
| Diego Alejandro del Valle Pimentel | A01747310 |
| José Santiago Rueda Antonio | A01794118 |

# Avance 3 v2 — BeMyEyes + PASTIS-R (Mejoras)

#### Sintesis
BeMyEyes se utiliza porque busca separar el entendimiento de la imagen en 2 modelos, un perceptor y un interpretador.
La utilidad de este _approach_ viene a que se reduce la utilización de memoria por parte de todo el proceso
al utilizar una herramienta secundaria, en este caso GPT 4o, de forma que solo hay que preocuparse por el
procesamiento de imagen y el envio de esta por medio de un API.
En este caso la interpretabilidad si es importante debido a que se busca que el _reasoner_ pueda interpretar 
la información que se le comparte.


**Cambios respecto a v1:**

| # | Mejora | Origen |
|---|--------|--------|
| 1 | Multi-turn conversation (2-3 turnos) | Paper BeMyEyes Table 4 ablation |
| 2 | NDVI temporal como texto al Reasoner | Literatura crop classification |
| 3 | Fuzzy name matching + aliases | Bug fix — parsing |
| 4 | Prompt Perceiver domain-specific | Best practice remote sensing VLMs |
| 5 | Guidelines Reasoner con fenología completa | Fenología cultivos franceses |
| 6 | Qwen2.5-VL-7B (4-bit) en vez de 3B | Paper BeMyEyes usa 7B |
| 7 | False color (NIR-R-G) como 2ª imagen | Espectral — NIR discrimina vegetación |
| 8 | SAR stats como texto | PASTIS-R benchmark |
| 9 | Evaluación estratificada n=300 | Validez estadística |

---
## 5. Implementación BeMyEyes v2

```
              ┌──────────────────┐
              │  Imagen RGB/FC   │
              │  (parcela)       │
              └───────┬──────────┘
                      │
                      ▼
         ┌────────────────────────┐    Turn 1: descripción
         │   PERCEIVER            │ ──────────────────────┐
         │   Qwen2.5-VL-7B (4bit)│                        │
         │   Local GPU            │ ◀──────────┐          │
         └────────────────────────┘  Turn 3:   │          ▼
                                     respuestas│  ┌───────────────────┐
                                               │  │   REASONER        │
              ┌──────────────────┐             │  │   GPT-4o (API)    │
              │  NDVI temporal   │ ───────────────│                   │
              │  SAR stats       │  (texto)    │  │   Turn 2: preguntas
              │  (texto)         │             └──│   Turn 4: clasif. │
              └──────────────────┘                └───────────────────┘
```

### 5A. Configuración

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import Counter
from difflib import get_close_matches
import json, os, tempfile

print("✅ Librerías base cargadas")

✅ Librerías base cargadas


In [ ]:
# === RUTAS ===
_PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "reports" else Path.cwd()
TRAIN_DATA_DIR = _PROJECT_ROOT / "train_data_v0"
PASTIS_PATH   = _PROJECT_ROOT / "dataset" / "PASTIS-R"

# === CLASES ===
CLASSES = {
    1: "Meadow", 2: "Soft_winter_wheat", 3: "Corn",
    4: "Winter_barley", 5: "Winter_rapeseed", 6: "Spring_barley",
    7: "Sunflower", 8: "Grapevine", 9: "Beet", 10: "Winter_triticale",
    11: "Winter_durum_wheat", 12: "Fruits_vegetables_flowers", 13: "Potatoes",
    14: "Leguminous_fodder", 15: "Soybeans", 16: "Orchard",
    17: "Mixed_cereal", 18: "Sorghum"
}

# === BANDAS S2 ===
# [B2, B3, B4, B5, B6, B7, B8, B8A, B11, B12]  → indices 0-9
S2_BANDS = {
    "B2_Blue": 0, "B3_Green": 1, "B4_Red": 2,
    "B5_RE1": 3, "B6_RE2": 4, "B7_RE3": 5,
    "B8_NIR": 6, "B8A_NIR2": 7, "B11_SWIR1": 8, "B12_SWIR2": 9
}
IDX_RED = 2
IDX_NIR = 6

# === API KEY ===
# Cargar desde .env o setear manualmente
from dotenv import load_dotenv
load_dotenv(override=True)
# Si no hay .env: os.environ['OPENAI_API_KEY'] = 'sk-...'

# === CHECKS ===
print(f"📁 Imágenes:  {TRAIN_DATA_DIR}  {'✅' if TRAIN_DATA_DIR.is_dir() else '❌'}")
print(f"📁 PASTIS-R:  {PASTIS_PATH}  {'✅' if PASTIS_PATH.is_dir() else '⚠️  (NDVI/SAR no disponibles)'}")
print(f"🌾 {len(CLASSES)} clases")

### 5A.1 Generación de `train_data_v0` (si no existe)

Misma lógica que v1: mediana temporal S2 → RGB, recorte por parcela, carpetas por clase.

In [ ]:
OUTPUT_DIR = TRAIN_DATA_DIR
MIN_PARCEL_PX = 8

def get_patch_ids(data_path):
    s2_dir = data_path / "DATA_S2"
    if not s2_dir.is_dir():
        return []
    return [int(f.stem.split("_")[1]) for f in sorted(s2_dir.glob("S2_*.npy"))]

def compute_median_rgb(s2):
    median = np.median(s2.astype(np.float32), axis=0)
    rgb = np.stack([median[2], median[1], median[0]], axis=0)
    rgb_norm = np.zeros((3, 128, 128), dtype=np.float32)
    for i in range(3):
        band = rgb[i]
        p2, p98 = np.percentile(band, [2, 98])
        if p98 > p2:
            rgb_norm[i] = np.clip((band - p2) / (p98 - p2), 0, 1)
    return (np.transpose(rgb_norm, (1, 2, 0)) * 255).astype(np.uint8)

def extract_parcels_from_patch(patch_id, data_path):
    s2_file  = data_path / "DATA_S2" / f"S2_{patch_id}.npy"
    ann_file = data_path / "ANNOTATIONS" / f"TARGET_{patch_id}.npy"
    inst_file = data_path / "INSTANCE_ANNOTATIONS" / f"INSTANCES_{patch_id}.npy"
    if not (s2_file.exists() and ann_file.exists() and inst_file.exists()):
        return []
    s2 = np.load(s2_file)
    semantic = np.load(ann_file)[0]
    instances = np.load(inst_file)
    rgb = compute_median_rgb(s2)
    saved = []
    for inst_id in np.unique(instances):
        if inst_id == 0:
            continue
        mask = instances == inst_id
        classes_in_parcel = semantic[mask]
        class_id = int(np.bincount(classes_in_parcel).argmax())
        if class_id < 1 or class_id > 18:
            continue
        rows = np.where(mask.any(axis=1))[0]
        cols = np.where(mask.any(axis=0))[0]
        r_min, r_max = rows[0], rows[-1] + 1
        c_min, c_max = cols[0], cols[-1] + 1
        h, w = r_max - r_min, c_max - c_min
        if h < MIN_PARCEL_PX or w < MIN_PARCEL_PX:
            continue
        crop = rgb[r_min:r_max, c_min:c_max]
        folder_name = CLASSES[class_id]
        out_dir = OUTPUT_DIR / folder_name
        out_dir.mkdir(parents=True, exist_ok=True)
        fname = f"p{patch_id}_i{inst_id}.png"
        Image.fromarray(crop).save(out_dir / fname)
        saved.append((class_id, patch_id, int(inst_id), h, w))
    return saved

if not TRAIN_DATA_DIR.is_dir() and PASTIS_PATH.is_dir():
    patch_ids = get_patch_ids(PASTIS_PATH)
    all_records = []
    for pid in tqdm(patch_ids, desc="Generando train_data_v0"):
        try:
            all_records.extend(extract_parcels_from_patch(pid, PASTIS_PATH))
        except Exception:
            pass
    print(f"✅ Generadas {len(all_records):,} imágenes en {TRAIN_DATA_DIR}")
elif TRAIN_DATA_DIR.is_dir():
    print("✅ train_data_v0 ya existe, se omite la generación.")
else:
    print("⚠️  No se encontró dataset/PASTIS-R ni train_data_v0.")

### 5B. Carga de imágenes

In [ ]:
def get_all_images():
    images = []
    for class_folder in TRAIN_DATA_DIR.iterdir():
        if not class_folder.is_dir():
            continue
        class_name = class_folder.name
        class_id = next((k for k, v in CLASSES.items() if v == class_name), None)
        if class_id is None:
            continue
        for img_path in class_folder.glob("*.png"):
            parts = img_path.stem.split('_')
            patch_id = int(parts[0][1:])
            instance_id = int(parts[1][1:])
            images.append({
                'path': img_path,
                'class_name': class_name,
                'class_id': class_id,
                'patch_id': patch_id,
                'instance_id': instance_id
            })
    return images

if not TRAIN_DATA_DIR.is_dir():
    print("⚠️  Ejecuta primero la generación de train_data_v0.")
    all_images = []
else:
    all_images = get_all_images()
    print(f"📦 Total imágenes: {len(all_images):,}")
    class_counts = Counter(img['class_name'] for img in all_images)
    print(f"\nDistribución de clases:")
    for cls, count in class_counts.most_common():
        print(f"   {cls}: {count:,}")

### 5B.1 Carga de features espectrales (NDVI temporal, SAR, False Color)

**Nuevo en v2**: Si PASTIS-R está disponible, extrae:
- Perfil NDVI temporal por parcela (la señal #1 para clasificar cultivos)
- Estadísticas SAR (VV, VH)
- Imagen false color (NIR-R-G)

Estas features van como **texto** al Reasoner y como **imagen adicional** al Perceiver.


#### Importancia de features
En este caso, no se utiliza algo como un PCA debido al hecho que se trabaja con un modelo no supervisado. Sin embargo, se utiliza NDVI
temporal para poder eliminar features al evaluar impacto de desempeño

In [ ]:
class SpectralFeatures:
    """Extrae NDVI temporal, SAR stats y false color de PASTIS-R."""

    def __init__(self, pastis_path):
        self.pastis_path = Path(pastis_path)
        self.available = self.pastis_path.is_dir()
        self._cache = {}  # cache de S2/S1/instances por patch_id
        if self.available:
            print("✅ SpectralFeatures: PASTIS-R encontrado")
        else:
            print("⚠️  SpectralFeatures: PASTIS-R no encontrado — features deshabilitadas")

    def _load_patch_data(self, patch_id):
        """Carga y cachea S2, instances y opcionalmente S1A para un patch."""
        if patch_id in self._cache:
            return self._cache[patch_id]

        data = {"s2": None, "s1a": None, "instances": None}
        s2_file = self.pastis_path / "DATA_S2" / f"S2_{patch_id}.npy"
        inst_file = self.pastis_path / "INSTANCE_ANNOTATIONS" / f"INSTANCES_{patch_id}.npy"
        s1a_file = self.pastis_path / "DATA_S1A" / f"S1A_{patch_id}.npy"

        if s2_file.exists():
            data["s2"] = np.load(s2_file)
        if inst_file.exists():
            data["instances"] = np.load(inst_file)
        if s1a_file.exists():
            data["s1a"] = np.load(s1a_file)

        self._cache[patch_id] = data
        return data

    def get_parcel_mask(self, patch_id, instance_id):
        """Retorna máscara booleana (128,128) de la parcela."""
        data = self._load_patch_data(patch_id)
        if data["instances"] is None:
            return None
        return data["instances"] == instance_id

    def get_ndvi_profile(self, patch_id, instance_id):
        """Calcula perfil NDVI temporal promedio de la parcela.

        Returns: list of floats o None si no hay datos.
        """
        if not self.available:
            return None
        data = self._load_patch_data(patch_id)
        if data["s2"] is None or data["instances"] is None:
            return None

        mask = data["instances"] == instance_id
        if mask.sum() == 0:
            return None

        s2 = data["s2"].astype(np.float32)  # (T, 10, 128, 128)
        nir = s2[:, IDX_NIR, :, :]  # (T, 128, 128)
        red = s2[:, IDX_RED, :, :]

        denom = nir + red + 1e-8
        ndvi = (nir - red) / denom  # (T, 128, 128)

        # Promedio espacial dentro de la parcela por fecha
        profile = []
        for t in range(ndvi.shape[0]):
            vals = ndvi[t][mask]
            profile.append(round(float(np.nanmean(vals)), 3))
        return profile

    def get_sar_stats(self, patch_id, instance_id):
        """Estadísticas SAR promedio de la parcela.

        Returns: dict con VV_mean, VH_mean, VV_VH_ratio o None.
        """
        if not self.available:
            return None
        data = self._load_patch_data(patch_id)
        if data["s1a"] is None or data["instances"] is None:
            return None

        mask = data["instances"] == instance_id
        if mask.sum() == 0:
            return None

        s1a = data["s1a"].astype(np.float32)  # (T, 3, 128, 128)
        # Mediana temporal
        s1_median = np.median(s1a, axis=0)  # (3, 128, 128)
        vv = s1_median[0][mask]
        vh = s1_median[1][mask]
        return {
            "VV_mean": round(float(np.mean(vv)), 2),
            "VH_mean": round(float(np.mean(vh)), 2),
            "VV_VH_ratio": round(float(np.mean(vv) / (np.mean(vh) + 1e-8)), 2)
        }

    def get_false_color_path(self, patch_id, instance_id):
        """Genera imagen false color (NIR-R-G) recortada a la parcela.

        Returns: path a imagen temporal PNG o None.
        """
        if not self.available:
            return None
        data = self._load_patch_data(patch_id)
        if data["s2"] is None or data["instances"] is None:
            return None

        mask = data["instances"] == instance_id
        if mask.sum() == 0:
            return None

        s2 = data["s2"].astype(np.float32)
        median = np.median(s2, axis=0)  # (10, 128, 128)

        # False color: NIR(B8) - Red(B4) - Green(B3)
        fc = np.stack([median[IDX_NIR], median[IDX_RED], median[S2_BANDS["B3_Green"]]], axis=0)
        fc_norm = np.zeros_like(fc)
        for i in range(3):
            band = fc[i]
            p2, p98 = np.percentile(band, [2, 98])
            if p98 > p2:
                fc_norm[i] = np.clip((band - p2) / (p98 - p2), 0, 1)
        fc_uint8 = (np.transpose(fc_norm, (1, 2, 0)) * 255).astype(np.uint8)

        # Recortar al bounding box de la parcela
        rows = np.where(mask.any(axis=1))[0]
        cols = np.where(mask.any(axis=0))[0]
        if len(rows) == 0 or len(cols) == 0:
            return None
        r_min, r_max = rows[0], rows[-1] + 1
        c_min, c_max = cols[0], cols[-1] + 1
        crop = fc_uint8[r_min:r_max, c_min:c_max]

        # Guardar en archivo temporal
        tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
        Image.fromarray(crop).save(tmp.name)
        return tmp.name

    def format_ndvi_text(self, profile):
        """Formatea perfil NDVI como texto legible para el Reasoner."""
        if profile is None:
            return "NDVI temporal: no disponible"

        peak_val = max(profile)
        peak_idx = profile.index(peak_val)
        min_val = min(profile)
        n = len(profile)

        # Estimar mes del peak (asumiendo Sep 2018 - Oct 2019, ~38-61 fechas)
        # Aproximación: dividir en 14 meses
        months = ["Sep18","Oct18","Nov18","Dec18","Jan19","Feb19","Mar19",
                   "Apr19","May19","Jun19","Jul19","Aug19","Sep19","Oct19"]
        if n > 0:
            peak_month = months[min(int(peak_idx / n * len(months)), len(months) - 1)]
        else:
            peak_month = "unknown"

        # Subsampling para prompt (máx 20 valores)
        step = max(1, n // 20)
        sampled = [profile[i] for i in range(0, n, step)]

        lines = [
            f"NDVI temporal profile ({n} dates, Sep 2018 - Oct 2019):",
            f"  Sampled values: {sampled}",
            f"  Peak NDVI: {peak_val:.3f} at date {peak_idx}/{n} (~{peak_month})",
            f"  Min NDVI:  {min_val:.3f}",
            f"  Mean NDVI: {np.mean(profile):.3f}",
        ]

        # Detectar si hay green-up invernal (cultivo de invierno)
        if n >= 10:
            first_quarter = np.mean(profile[:n//4])
            last_quarter = np.mean(profile[-n//4:])
            mid = np.mean(profile[n//3:2*n//3])
            if first_quarter > 0.3 and mid > first_quarter:
                lines.append("  Pattern: vegetation present in early season (possible winter crop)")
            elif first_quarter < 0.2 and mid > 0.4:
                lines.append("  Pattern: bare soil early, growth mid-season (possible spring/summer crop)")

        return "\n".join(lines)

    def format_sar_text(self, sar_stats):
        """Formatea SAR stats como texto."""
        if sar_stats is None:
            return "SAR stats: no disponible"
        return (
            f"SAR Sentinel-1 (temporal median):\n"
            f"  VV mean: {sar_stats['VV_mean']:.2f}\n"
            f"  VH mean: {sar_stats['VH_mean']:.2f}\n"
            f"  VV/VH ratio: {sar_stats['VV_VH_ratio']:.2f}"
        )


# Instanciar
spectral = SpectralFeatures(PASTIS_PATH)

### 5C. Perceiver — Qwen2.5-VL-7B (Local, 4-bit)

In [ ]:
PERCEIVER_PROMPT_V2 = (
    "You are a remote sensing analyst examining a Sentinel-2 satellite image of an "
    "agricultural parcel in France. The person receiving your description CANNOT see "
    "the image and must classify the crop type based solely on your description.\n\n"
    "Describe with precision:\n"
    "1. DOMINANT COLOR: Exact shade (dark green, lime green, golden yellow, brown, "
    "bright yellow). What percentage of the parcel is each color?\n"
    "2. ROW STRUCTURE: Are there visible parallel rows? Estimate spacing "
    "(tight <2m, medium 2-5m, wide >5m). Row direction?\n"
    "3. CANOPY COVERAGE: What % of ground is covered by vegetation vs bare soil?\n"
    "4. TEXTURE: Smooth/uniform, coarse, grainy, striped, dotted (individual plants)\n"
    "5. WITHIN-PARCEL VARIATION: Homogeneous or patches of different color/texture?\n"
    "6. SIZE: Approximate parcel dimensions in pixels.\n\n"
    "Be quantitative. Say 'approximately 70% dark green with 30% brown soil visible "
    "between rows' not just 'green field'."
)

print("✅ Prompt Perceiver v2 definido")

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

class Perceiver:
    """
    Perceiver v2: Qwen2.5-VL-7B-Instruct (4-bit NF4).
    Soporta describe() y answer() para multi-turn.
    """

    def __init__(self, model_name="Qwen/Qwen2.5-VL-7B-Instruct", max_new_tokens=256):
        print(f"Cargando {model_name}...")

        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_name,
            quantization_config=bnb,
            device_map="auto",
        ).eval()

        self.processor = AutoProcessor.from_pretrained(model_name)
        self.max_new_tokens = max_new_tokens
        self.prompt = PERCEIVER_PROMPT_V2

        device = next(self.model.parameters()).device
        vram = torch.cuda.memory_allocated() / 1e9
        print(f"✅ Perceiver cargado en {device} | 4bit NF4 | VRAM: {vram:.1f} GB")

    @torch.inference_mode()
    def _generate(self, messages):
        """Genera respuesta a partir de mensajes Qwen-VL."""
        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        )
        device = next(self.model.parameters()).device
        inputs = {k: (v.to(device) if hasattr(v, "to") else v) for k, v in inputs.items()}

        out = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            do_sample=False,
        )
        out = out[:, inputs["input_ids"].shape[1]:]
        return self.processor.batch_decode(out, skip_special_tokens=True)[0].strip()

    def describe(self, img_path, fc_path=None):
        """Turn 1: Descripción inicial de la imagen.

        Args:
            img_path: Ruta a imagen RGB.
            fc_path: Ruta opcional a imagen false color (NIR-R-G).
        """
        content = [{"type": "image", "image": str(img_path)}]

        if fc_path is not None:
            content.append({"type": "image", "image": str(fc_path)})
            user_text = (
                "Image 1 is true-color RGB. Image 2 is false-color (NIR-Red-Green) "
                "where bright red indicates active vegetation. Describe both images."
            )
        else:
            user_text = "Describe this satellite image of an agricultural parcel."

        content.append({"type": "text", "text": user_text})

        messages = [
            {"role": "system", "content": [{"type": "text", "text": self.prompt}]},
            {"role": "user", "content": content}
        ]
        return self._generate(messages)

    def answer(self, img_path, questions, fc_path=None):
        """Turn 3: Responde preguntas específicas del Reasoner.

        Args:
            img_path: Ruta a imagen RGB.
            questions: str con las preguntas del Reasoner.
            fc_path: Ruta opcional a false color.
        """
        content = [{"type": "image", "image": str(img_path)}]
        if fc_path is not None:
            content.append({"type": "image", "image": str(fc_path)})

        content.append({
            "type": "text",
            "text": (
                "A crop classification expert has these follow-up questions about "
                "the satellite image. Answer each one precisely based on what you see:\n\n"
                f"{questions}"
            )
        })

        messages = [
            {"role": "system", "content": [{"type": "text", "text": self.prompt}]},
            {"role": "user", "content": content}
        ]
        return self._generate(messages)

In [ ]:
# === CARGAR PERCEIVER ===
# RTX 5080 16GB: 7B en 4-bit usa ~5-6GB → cabe sin problema
# Si tienes problemas de VRAM, cambia a "Qwen/Qwen2.5-VL-3B-Instruct"
perceiver = Perceiver(model_name="Qwen/Qwen2.5-VL-7B-Instruct")

In [ ]:
# Test Perceiver v2
if all_images:
    test_img = all_images[0]
    plt.figure(figsize=(4, 4))
    plt.imshow(Image.open(test_img['path']))
    plt.title(f"GT: {test_img['class_name']}")
    plt.axis('off')
    plt.show()

    desc = perceiver.describe(test_img['path'])
    print("📸 PERCEIVER (v2):")
    print(desc)

### 5D. Reasoner — GPT-4o (API)

In [ ]:
CROP_LIST = "\n".join([f"- {v.replace('_', ' ')}" for v in CLASSES.values()])

REASONER_SYSTEM = f"""You are an expert agronomist classifying French crops from Sentinel-2 descriptions and spectral data.

You CANNOT see images. You receive:
1. Visual description from a perception agent
2. NDVI temporal profile (if available)
3. SAR radar statistics (if available)
4. Answers to your follow-up questions

=== AVAILABLE CROPS ===
{CROP_LIST}

=== PHENOLOGY-BASED CLASSIFICATION RULES ===

WINTER CROPS (sown Oct-Nov, green through winter, harvest Jun-Jul):
- Soft winter wheat: Green in winter, golden by June. Tight row pattern. NDVI peak ~May.
- Winter barley: Like wheat but earlier maturity. NDVI peak ~April-May.
- Winter rapeseed: DISTINCTIVE bright yellow flowers in April. Very high NDVI in spring. Dense canopy.
- Winter triticale: Like wheat, slightly later maturity. Rare.
- Winter durum wheat: Like soft wheat but drier areas, sparser canopy.

SPRING/SUMMER CROPS (sown Mar-May, harvest Sep-Oct):
- Corn: Strong parallel WIDE rows (~75cm). Very dense dark green. NDVI peak Jul-Aug. Tall canopy.
- Sunflower: Wide rows (~50cm), can show yellow in Jul. NDVI peak Jul.
- Spring barley: Like winter barley but sown March. NDVI peak Jun-Jul.
- Soybeans: Medium rows, medium green. NDVI peak Jul-Aug.
- Sorghum: Like corn but finer texture, narrower rows. NDVI peak Aug.
- Beet: Very dense, uniform dark green, NO visible rows. NDVI peak Jul-Aug.
- Potatoes: Medium rows, medium green. Earlier senescence.

PERENNIAL / OTHER:
- Meadow: Uniform green year-round. NO rows. Moderate stable NDVI.
- Grapevine: Clear parallel rows with WIDE gaps, bare soil visible between rows. Low NDVI.
- Orchard: Grid of individual trees/dots. Low-moderate NDVI.
- Fruits vegetables flowers: Highly variable, often small parcels.
- Leguminous fodder: Like meadow but may show row structure. Alfalfa-like.
- Mixed cereal: Mix of cereal textures.

=== DECISION KEYS ===
1. Rows visible? NO → Meadow, Beet, or Fruits/veg
2. Row spacing WIDE with bare soil? → Grapevine, Corn, Sunflower
3. NDVI peak in winter/early spring? → Winter crop
4. NDVI peak in summer? → Spring/summer crop
5. Bright yellow color? → Rapeseed (spring) or Sunflower (summer)
6. Grid of dots? → Orchard"""

REASONER_CLASSIFY_PROMPT = """Based on ALL available evidence, classify the crop.

Respond EXACTLY in this format (no extra text):
CROP: [exact crop name from the list]
CONFIDENCE: [high/medium/low]
REASON: [2-3 sentences citing specific visual + temporal evidence]"""

REASONER_QUESTIONS_PROMPT = """Based on the visual description and any spectral data provided, generate 2-3 SHORT follow-up questions to help disambiguate the crop type.

Focus on the most discriminative features you still need:
- Row spacing and pattern details
- Specific colors or color changes
- Canopy density and uniformity
- Parcel size

Format: numbered list, max 3 questions."""

print("✅ Prompts Reasoner v2 definidos")

In [ ]:
from openai import OpenAI

class Reasoner:
    """Reasoner v2: GPT-4o con multi-turn (preguntas + clasificación)."""

    def __init__(self, model="gpt-4o"):
        self.client = OpenAI()
        self.model = model
        print(f"✅ Reasoner configurado ({self.model})")

    def _call(self, messages, max_tokens=200):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            max_tokens=max_tokens,
            temperature=0,
        )
        return response.choices[0].message.content

    def ask_questions(self, description, ndvi_text="", sar_text=""):
        """Turn 2: Genera preguntas de clarificación."""
        user_content = f"Visual description:\n{description}"
        if ndvi_text:
            user_content += f"\n\n{ndvi_text}"
        if sar_text:
            user_content += f"\n\n{sar_text}"

        messages = [
            {"role": "system", "content": REASONER_SYSTEM},
            {"role": "user", "content": user_content},
            {"role": "user", "content": REASONER_QUESTIONS_PROMPT},
        ]
        return self._call(messages, max_tokens=200)

    def classify(self, description, ndvi_text="", sar_text="",
                 questions="", answers=""):
        """Turn 4: Clasificación final con todo el contexto."""
        context_parts = [f"Visual description:\n{description}"]
        if ndvi_text:
            context_parts.append(ndvi_text)
        if sar_text:
            context_parts.append(sar_text)
        if questions and answers:
            context_parts.append(f"Follow-up questions:\n{questions}")
            context_parts.append(f"Answers from perception agent:\n{answers}")

        full_context = "\n\n".join(context_parts)

        messages = [
            {"role": "system", "content": REASONER_SYSTEM},
            {"role": "user", "content": full_context},
            {"role": "user", "content": REASONER_CLASSIFY_PROMPT},
        ]
        return self._call(messages, max_tokens=150)

In [ ]:
reasoner = Reasoner()

### 5E. Name Matching (Fuzzy + Aliases)

In [ ]:
CLASS_ALIASES = {}

# Construir aliases automáticos desde CLASSES
for cid, cname in CLASSES.items():
    # Nombre normalizado (lowercase, sin underscores)
    norm = cname.lower().replace("_", " ")
    CLASS_ALIASES[norm] = cname

    # Variantes comunes
    words = norm.split()
    if len(words) > 1:
        CLASS_ALIASES[words[-1]] = cname  # última palabra: "wheat", "barley", etc.

# Aliases manuales para casos conocidos
_MANUAL = {
    "wheat": "Soft_winter_wheat",  # default si solo dice "wheat"
    "winter wheat": "Soft_winter_wheat",
    "soft wheat": "Soft_winter_wheat",
    "maize": "Corn",
    "vineyard": "Grapevine",
    "vine": "Grapevine",
    "vines": "Grapevine",
    "grape": "Grapevine",
    "canola": "Winter_rapeseed",
    "rapeseed": "Winter_rapeseed",
    "rape": "Winter_rapeseed",
    "sugar beet": "Beet",
    "sugarbeet": "Beet",
    "potato": "Potatoes",
    "soybean": "Soybeans",
    "soy": "Soybeans",
    "alfalfa": "Leguminous_fodder",
    "fodder": "Leguminous_fodder",
    "clover": "Leguminous_fodder",
    "triticale": "Winter_triticale",
    "durum": "Winter_durum_wheat",
    "durum wheat": "Winter_durum_wheat",
    "orchard trees": "Orchard",
    "fruit trees": "Orchard",
    "grassland": "Meadow",
    "pasture": "Meadow",
    "grass": "Meadow",
    "fruits vegetables flowers": "Fruits_vegetables_flowers",
    "fruits/veg/flowers": "Fruits_vegetables_flowers",
    "fruits and vegetables": "Fruits_vegetables_flowers",
    "mixed cereals": "Mixed_cereal",
}
CLASS_ALIASES.update(_MANUAL)

# Lista de nombres canónicos para fuzzy matching
_CANONICAL_KEYS = list(CLASS_ALIASES.keys())


def match_prediction(pred_text):
    """Mapea la predicción del Reasoner a un nombre canónico de CLASSES.

    1. Exact match (lowercase)
    2. Fuzzy match (difflib, cutoff=0.6)
    3. 'Unknown' si no hay match
    """
    if not pred_text or not pred_text.strip():
        return "Unknown"

    normalized = pred_text.lower().strip().rstrip(".")

    # 1. Exact
    if normalized in CLASS_ALIASES:
        return CLASS_ALIASES[normalized]

    # 2. Fuzzy
    matches = get_close_matches(normalized, _CANONICAL_KEYS, n=1, cutoff=0.6)
    if matches:
        return CLASS_ALIASES[matches[0]]

    return "Unknown"


# Quick test
assert match_prediction("Vineyard") == "Grapevine"
assert match_prediction("Soft winter wheat") == "Soft_winter_wheat"
assert match_prediction("CORN") == "Corn"
assert match_prediction("sugar beet") == "Beet"
assert match_prediction("maize") == "Corn"
print("✅ Name matching: todos los tests pasan")

### 5F. Orquestador Multi-Turn

Pipeline completo:
1. **Perceiver** describe imagen (+ false color si disponible)
2. **Reasoner** analiza descripción + NDVI + SAR → genera preguntas
3. **Perceiver** responde preguntas con la imagen
4. **Reasoner** clasifica con todo el contexto

In [ ]:
def parse_response(response):
    """Extrae CROP, CONFIDENCE, REASON del response del Reasoner."""
    result = {'crop': 'Unknown', 'confidence': 'low', 'reason': ''}
    for line in response.split('\n'):
        upper = line.upper().strip()
        if upper.startswith('CROP:'):
            result['crop'] = line.split(':', 1)[1].strip()
        elif upper.startswith('CONFIDENCE:'):
            result['confidence'] = line.split(':', 1)[1].strip().lower()
        elif upper.startswith('REASON:'):
            result['reason'] = line.split(':', 1)[1].strip()
    return result


def run_pipeline_v2(img_info, perceiver, reasoner, spectral, verbose=True):
    """Pipeline BeMyEyes v2 con multi-turn + features espectrales."""
    gt_class = img_info['class_name']
    patch_id = img_info['patch_id']
    instance_id = img_info['instance_id']

    if verbose:
        print(f"\n{'='*60}")
        print(f"Patch {patch_id}, Instance {instance_id} | GT: {gt_class}")
        print('='*60)

    # --- Features espectrales (texto para el Reasoner) ---
    ndvi_profile = spectral.get_ndvi_profile(patch_id, instance_id)
    ndvi_text = spectral.format_ndvi_text(ndvi_profile)
    sar_stats = spectral.get_sar_stats(patch_id, instance_id)
    sar_text = spectral.format_sar_text(sar_stats)
    fc_path = spectral.get_false_color_path(patch_id, instance_id)

    # --- Turn 1: Perceiver describe ---
    description = perceiver.describe(img_info['path'], fc_path=fc_path)
    if verbose:
        print(f"\n📸 T1 Perceiver:\n{description[:400]}...")

    # --- Turn 2: Reasoner genera preguntas ---
    questions = reasoner.ask_questions(description, ndvi_text, sar_text)
    if verbose:
        print(f"\n🧠 T2 Reasoner (preguntas):\n{questions}")

    # --- Turn 3: Perceiver responde ---
    answers = perceiver.answer(img_info['path'], questions, fc_path=fc_path)
    if verbose:
        print(f"\n📸 T3 Perceiver (respuestas):\n{answers[:400]}...")

    # --- Turn 4: Reasoner clasifica ---
    classification = reasoner.classify(
        description, ndvi_text, sar_text, questions, answers
    )
    if verbose:
        print(f"\n🧠 T4 Reasoner (clasificación):\n{classification}")

    # --- Parse + match ---
    parsed = parse_response(classification)
    matched = match_prediction(parsed['crop'])
    correct = matched == gt_class

    if verbose:
        status = "✅" if correct else "❌"
        print(f"\n{status} Pred: {parsed['crop']} → matched: {matched} | GT: {gt_class}")

    # Cleanup temp false color
    if fc_path is not None:
        try:
            os.unlink(fc_path)
        except OSError:
            pass

    return {
        'patch_id': patch_id,
        'instance_id': instance_id,
        'description': description,
        'questions': questions,
        'answers': answers,
        'raw_prediction': parsed['crop'],
        'prediction': matched,
        'confidence': parsed['confidence'],
        'reason': parsed['reason'],
        'ground_truth': gt_class,
        'correct': correct,
        'ndvi_available': ndvi_profile is not None,
        'sar_available': sar_stats is not None,
    }

print("✅ Pipeline v2 definido")

In [ ]:
# Test pipeline completo (1 imagen)
if all_images:
    result = run_pipeline_v2(all_images[0], perceiver, reasoner, spectral, verbose=True)

### 5G. Evaluación Estratificada

**Cambio v2**: Muestreo estratificado para que cada clase tenga representación mínima.

In [ ]:
def evaluate_v2(perceiver, reasoner, spectral, images,
                n_samples=300, min_per_class=5, seed=42):
    """Evaluación estratificada del pipeline v2.

    Args:
        n_samples: Total de muestras objetivo.
        min_per_class: Mínimo de muestras por clase.
    """
    np.random.seed(seed)

    # Agrupar por clase
    by_class = {}
    for i, img in enumerate(images):
        cls = img['class_name']
        if cls not in by_class:
            by_class[cls] = []
        by_class[cls].append(i)

    # Muestreo estratificado
    selected = []
    n_classes = len(by_class)
    per_class_target = max(min_per_class, n_samples // n_classes)

    for cls, indices in by_class.items():
        n_take = min(per_class_target, len(indices))
        chosen = np.random.choice(indices, n_take, replace=False)
        selected.extend(chosen.tolist())

    # Si sobran slots, rellenar aleatoriamente
    remaining = set(range(len(images))) - set(selected)
    if len(selected) < n_samples and remaining:
        extra = np.random.choice(list(remaining),
                                 min(n_samples - len(selected), len(remaining)),
                                 replace=False)
        selected.extend(extra.tolist())

    np.random.shuffle(selected)
    print(f"📊 Evaluando {len(selected)} muestras ({n_classes} clases)")

    # Ejecutar
    results = []
    for idx in tqdm(selected, desc="Evaluando"):
        try:
            result = run_pipeline_v2(images[idx], perceiver, reasoner, spectral,
                                     verbose=False)
            results.append(result)
        except Exception as e:
            print(f"  ⚠️ Error idx={idx}: {e}")

    # === Métricas ===
    correct = sum(r['correct'] for r in results)
    total = len(results)
    accuracy = correct / total if total > 0 else 0

    # Por clase
    class_correct = {}
    class_total = {}
    for r in results:
        gt = r['ground_truth']
        class_total[gt] = class_total.get(gt, 0) + 1
        if r['correct']:
            class_correct[gt] = class_correct.get(gt, 0) + 1

    class_acc = {}
    for cls in class_total:
        class_acc[cls] = class_correct.get(cls, 0) / class_total[cls]

    # Por confianza
    conf_correct = {}
    conf_total = {}
    for r in results:
        c = r['confidence']
        conf_total[c] = conf_total.get(c, 0) + 1
        if r['correct']:
            conf_correct[c] = conf_correct.get(c, 0) + 1

    # === Resumen ===
    print(f"\n{'='*60}")
    print(f"RESULTADOS v2")
    print(f"{'='*60}")
    print(f"Overall Accuracy: {accuracy:.1%} ({correct}/{total})")

    print(f"\nPor clase:")
    for cls, acc in sorted(class_acc.items(), key=lambda x: -x[1]):
        n = class_total[cls]
        c = class_correct.get(cls, 0)
        bar = "█" * int(acc * 20) + "░" * (20 - int(acc * 20))
        print(f"  {cls:30s} {bar} {acc:5.0%} ({c}/{n})")

    print(f"\nPor confianza:")
    for conf in ["high", "medium", "low"]:
        if conf in conf_total:
            ct = conf_total[conf]
            cc = conf_correct.get(conf, 0)
            print(f"  {conf:8s}: {cc/ct:.0%} ({cc}/{ct})")

    # Features espectrales disponibles
    n_ndvi = sum(1 for r in results if r.get('ndvi_available'))
    n_sar = sum(1 for r in results if r.get('sar_available'))
    print(f"\nFeatures espectrales: NDVI={n_ndvi}/{total}, SAR={n_sar}/{total}")

    metrics = {
        'accuracy': accuracy,
        'class_accuracy': class_acc,
        'n_samples': total,
        'n_correct': correct,
    }
    return results, metrics

#### Análisis de métrica

En este caso se busca darle mayor importancia al _accuracy_ debido a que se considera que lo importante es ver que tan bueno es el modelo
para poder elegir correctamente la categoria elegida sin tener la supervisión para diferenciarlas

In [ ]:
# === EJECUTAR EVALUACIÓN ===
# Ajusta n_samples según tu presupuesto de API:
#   n=50  → ~$1-2 USD en GPT-4o, ~10 min
#   n=300 → ~$8-12 USD, ~60 min
#   n=500 → ~$15-20 USD, ~100 min

results, metrics = evaluate_v2(
    perceiver, reasoner, spectral, all_images,
    n_samples=50,       # empezar con 50, subir a 300 cuando estés listo
    min_per_class=2,
)

### 5H. Análisis de Errores

In [ ]:
# Confusiones más frecuentes
errors = [(r['ground_truth'], r['prediction'], r.get('raw_prediction',''))
          for r in results if not r['correct']]

error_pairs = Counter([(gt, pred) for gt, pred, _ in errors])
print("Confusiones más frecuentes (GT → Pred):")
for (gt, pred), count in error_pairs.most_common(10):
    print(f"  {gt:30s} → {pred:30s}  x{count}")

# Predicciones que no matchearon (Unknown)
unknowns = [(r['raw_prediction'], r['ground_truth'])
            for r in results if r['prediction'] == 'Unknown']
if unknowns:
    print(f"\n⚠️  {len(unknowns)} predicciones no matchearon:")
    for raw, gt in unknowns[:10]:
        print(f"  Raw: '{raw}' | GT: {gt}")

# Accuracy por disponibilidad de NDVI
with_ndvi = [r for r in results if r.get('ndvi_available')]
without_ndvi = [r for r in results if not r.get('ndvi_available')]
if with_ndvi:
    acc_w = sum(r['correct'] for r in with_ndvi) / len(with_ndvi)
    print(f"\nAccuracy CON NDVI:  {acc_w:.1%} (n={len(with_ndvi)})")
if without_ndvi:
    acc_wo = sum(r['correct'] for r in without_ndvi) / len(without_ndvi)
    print(f"Accuracy SIN NDVI:  {acc_wo:.1%} (n={len(without_ndvi)})")

In [ ]:
output = {
    'config': {
        'version': 'v2',
        'perceiver': 'Qwen2.5-VL-7B-Instruct (4-bit)',
        'reasoner': 'GPT-4o',
        'multi_turn': True,
        'ndvi_temporal': spectral.available,
        'sar_stats': spectral.available,
        'false_color': spectral.available,
        'n_samples': len(results),
    },
    'metrics': {
        'accuracy': round(metrics['accuracy'], 4),
        'class_accuracy': {k: round(v, 4) for k, v in metrics['class_accuracy'].items()},
    },
    'results': [
        {
            'patch_id': r['patch_id'],
            'ground_truth': r['ground_truth'],
            'raw_prediction': r.get('raw_prediction', ''),
            'prediction': r['prediction'],
            'confidence': r['confidence'],
            'correct': r['correct'],
        }
        for r in results
    ]
}

with open('bemyeyes_results_v2.json', 'w') as f:
    json.dump(output, f, indent=2)

print(f"✅ Resultados guardados en bemyeyes_results_v2.json")
print(f"   Accuracy: {metrics['accuracy']:.1%}")

#### Analisis de errores

Se busca llegar a una medida del desempeño basado en el _accuracy_ y a su vez los errores que se obtuvieron al revisar las predicciones.

---
## 6. Conclusiones — BeMyEyes v2

### 6.1 Cambios implementados

| Componente | v1 | v2 |
|------------|----|----|
| Perceiver | Qwen2.5-VL-3B (4-bit) | Qwen2.5-VL-7B (4-bit) |
| Prompt Perceiver | Genérico | Domain-specific remote sensing |
| Reasoner guidelines | 7/18 clases | 18/18 con fenología |
| Conversación | Single-turn | Multi-turn (4 pasos) |
| Info espectral | Solo RGB mediana | +NDVI temporal +SAR +False color |
| Name matching | String exacto | Aliases + fuzzy matching |
| Evaluación | Random n=50 | Estratificada por clase |

### 6.2 VRAM (RTX 5080 16GB)

| Componente | VRAM |
|------------|------|
| Qwen2.5-VL-7B (4-bit NF4) | ~5-6 GB |
| Inference peak | ~7-8 GB |
| **Libre** | **~8-9 GB** |

### 6.3 Limitaciones restantes

| Limitación | Posible mitigación |
|------------|-------------------|
| Sin SFT del Perceiver | Generar datos sintéticos con GPT-4o para fine-tune |
| Imagen estática (mediana) | Multi-epoch: invierno + primavera + verano |
| 1 solo run | Majority vote (3 runs) |